In [ ]:
!pip install transformers datasets torchmetrics

In [ ]:
import torch
import numpy as np
from transformers import AutoModelForAudioClassification, AutoFeatureExtractor
from datasets import load_dataset, Audio, DatasetDict
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, recall_score, roc_auc_score
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model_or = AutoModelForAudioClassification.from_pretrained("openai/whisper-base").to(device).eval()
model_ft = AutoModelForAudioClassification.from_pretrained("VasilisAsim/whisper-finetuned-TESS").to(device).eval()
feature_extractor = AutoFeatureExtractor.from_pretrained("VasilisAsim/whisper-finetuned-TESS")

In [ ]:
def load_tess_speaker_independent():
    dataset = load_dataset("AbstractTTS/TESS", split="train")
    dataset = dataset.cast_column("audio", Audio(decode=False))

    def get_speaker(example):
        filename = example["audio"]["path"]
        if "OAF" in filename:
            return "OAF"
        elif "YAF" in filename:
            return "YAF"
        return "unknown"

    train_dataset = dataset.filter(lambda x: get_speaker(x) == "OAF")
    test_dataset = dataset.filter(lambda x: get_speaker(x) == "YAF")
    train_dataset = train_dataset.cast_column("audio", Audio(sampling_rate=16_000))
    test_dataset = test_dataset.cast_column("audio", Audio(sampling_rate=16_000))
    return DatasetDict({"train": train_dataset, "test": test_dataset})

dataset = load_tess_speaker_independent()

In [ ]:
labels_sorted = sorted(set(dataset["train"]["emotion"]))
label2id = {l: i for i, l in enumerate(labels_sorted)}
id2label = {i: l for l, i in label2id.items()}
num_labels = len(labels_sorted)

In [ ]:
def extract_all_layers(model, ds, label_field="emotion"):
    per_layer_embeddings = None
    labels = []

    with torch.no_grad():
        for sample in ds:
            audio = sample["audio"]
            inputs = feature_extractor(audio["array"], sampling_rate=audio["sampling_rate"], return_tensors="pt")
            input_features = inputs["input_features"].to(device)

            outputs = model(input_features=input_features, output_hidden_states=True)
            hidden_states = outputs.hidden_states  # 7 στοιχεία: embedding output + 6 encoder layers

            if per_layer_embeddings is None:
                per_layer_embeddings = [[] for _ in hidden_states]

            for layer_idx, h in enumerate(hidden_states):
                per_layer_embeddings[layer_idx].append(h.mean(dim=1).squeeze().cpu().numpy())

            labels.append(label2id[sample[label_field]])

    per_layer_embeddings = [np.array(layer) for layer in per_layer_embeddings]
    return per_layer_embeddings, np.array(labels)

In [ ]:
def run_layer_wise_probing(model, model_name):
    print(f"Εξαγωγή embeddings ({model_name}) - train...")
    train_layers, train_labels = extract_all_layers(model, dataset["train"], label_field="emotion")
    print(f"Εξαγωγή embeddings ({model_name}) - test...")
    test_layers, test_labels = extract_all_layers(model, dataset["test"], label_field="emotion")

    results = {"accuracy": [], "macro_f1": [], "uar": [], "macro_auc": []}

    for layer_idx in range(len(train_layers)):
        X_train, X_test = train_layers[layer_idx], test_layers[layer_idx]

        clf = LogisticRegression(max_iter=2000)
        clf.fit(X_train, train_labels)

        preds = clf.predict(X_test)
        probs = clf.predict_proba(X_test)

        acc = accuracy_score(test_labels, preds)
        macro_f1 = f1_score(test_labels, preds, average="macro")
        uar = recall_score(test_labels, preds, average="macro")

        y_test_bin = label_binarize(test_labels, classes=list(range(num_labels)))
        macro_auc = roc_auc_score(y_test_bin, probs, average="macro", multi_class="ovr")

        results["accuracy"].append(acc)
        results["macro_f1"].append(macro_f1)
        results["uar"].append(uar)
        results["macro_auc"].append(macro_auc)

        print(f"  Layer {layer_idx}: acc={acc:.3f}  macro_f1={macro_f1:.3f}  uar={uar:.3f}  macro_auc={macro_auc:.3f}")

    return results

In [ ]:
results_or = run_layer_wise_probing(model_or, "Pretrained Whisper")
results_ft = run_layer_wise_probing(model_ft, "Fine-tuned Whisper")

In [ ]:
metrics = ["accuracy", "macro_f1", "uar", "macro_auc"]
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
for ax, metric in zip(axes.flat, metrics):
    layers = range(len(results_or[metric]))
    ax.plot(layers, results_or[metric], marker="o", label="Pretrained")
    ax.plot(layers, results_ft[metric], marker="o", label="Fine-tuned")
    ax.set_title(metric); ax.set_xlabel("Layer"); ax.set_ylabel(metric); ax.legend()

plt.suptitle("Layer-wise Linear Probing - Whisper + TESS")
plt.tight_layout()
plt.show()